In [ ]:
# IMPORT THƯ VIỆN

import os
import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
 
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
 
import timm
 
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from pathlib import Path
 
print("Import hoàn tất.")
print(f"PyTorch version : {torch.__version__}")
print(f"timm version    : {timm.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")

In [ ]:
# CẤU HÌNH
# Tập trung toàn bộ siêu tham số vào một chỗ để dễ điều chỉnh.
# Khác ResNet50: LR nhỏ hơn (Transformer nhạy cảm hơn với LR lớn),
# warm-up dài hơn, chỉ 2 giai đoạn thay vì 3.
 
SPLIT_PATH = "/kaggle/input/datasets/dunnguynvn/blood-data-l2/dataset_split"
OUTPUT_DIR = "/kaggle/working/swin_output"
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
 
# Đường dẫn đến ResNet50
RESNET_REPORT_PATH = "/kaggle/input/resnet50-output/classification_report.csv"
 
CLASSES = [
    "BA", "BNE", "EO", "ERB", "IG", "LY", "MMY",
    "MO", "MY", "MYELOBLAST", "PLATELET", "PMY", "SNE"
]
NUM_CLASSES = len(CLASSES)
 
IMG_SIZE    = 224
BATCH_SIZE  = 32
NUM_WORKERS = 2
PIN_MEMORY  = True
 
# 2 giai đoạn
EPOCHS_WARMUP   = 5     # Freeze backbone, train head
EPOCHS_FINETUNE = 25    # Unfreeze stage2 + stage3
TOTAL_EPOCHS    = EPOCHS_WARMUP + EPOCHS_FINETUNE  # = 30
 
LR_HEAD      = 1e-3    # LR cho classifier head
LR_BACKBONE  = 1e-5    
 
WEIGHT_DECAY        = 1e-4
LABEL_SMOOTHING     = 0.1
FOCAL_GAMMA         = 2.0
EARLY_STOP_PATIENCE = 7
RANDOM_SEED         = 42
 
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
 
print(f"\nCấu hình:")
print(f"  Device       : {DEVICE}")
print(f"  Num classes  : {NUM_CLASSES}")
print(f"  Total epochs : {TOTAL_EPOCHS} "
      f"(warmup={EPOCHS_WARMUP}, finetune={EPOCHS_FINETUNE})")
print(f"  Batch size   : {BATCH_SIZE}")
print(f"  Output dir   : {OUTPUT_DIR}")

In [ ]:
# DATA TRANSFORMS
# Val/Test: chỉ resize + normalize, không augment.
 
transform_train = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.1, contrast=0.1,
                           saturation=0.1, hue=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])
 
transform_val = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

In [ ]:
# DATASET & DATALOADER
# Dùng shuffle=True bình thường, bỏ WeightedRandomSampler.
# Focal Loss đã xử lý mất cân bằng qua gradient sau augmentation
# offline IG có 1000 ảnh nên xác suất thiếu IG trong batch rất thấp.
 
train_dataset = datasets.ImageFolder(
    root=os.path.join(SPLIT_PATH, "train"), transform=transform_train)
val_dataset   = datasets.ImageFolder(
    root=os.path.join(SPLIT_PATH, "val"),   transform=transform_val)
test_dataset  = datasets.ImageFolder(
    root=os.path.join(SPLIT_PATH, "test"),  transform=transform_val)
 
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                          shuffle=True,  num_workers=NUM_WORKERS,
                          pin_memory=PIN_MEMORY)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=NUM_WORKERS,
                          pin_memory=PIN_MEMORY)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=NUM_WORKERS,
                          pin_memory=PIN_MEMORY)
 
print(f"Dataset đã tải:")
print(f"  Train : {len(train_dataset)} ảnh")
print(f"  Val   : {len(val_dataset)} ảnh")
print(f"  Test  : {len(test_dataset)} ảnh")

In [ ]:
# FOCAL LOSS
# Giữ nguyên từ pipeline ResNet50 để đảm bảo consistency khi so sánh.
# Không dùng class_weight, không WeightedSampler.
 
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, label_smoothing=0.0):
        super().__init__()
        self.gamma           = gamma
        self.label_smoothing = label_smoothing
 
    def forward(self, inputs, targets):
        ce_loss = nn.CrossEntropyLoss(
            label_smoothing=self.label_smoothing,
            reduction='none'
        )(inputs, targets)
        pt    = torch.exp(-ce_loss)
        focal = (1 - pt) ** self.gamma * ce_loss
        return focal.mean()
 

In [ ]:
# MIXUP AUGMENTATION
# Giữ nguyên từ pipeline ResNet50.
# Không dùng ở giai đoạn Warm-up vì head chưa ổn định.
 
def mixup_data(x, y, alpha=0.4):
    lam     = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    index   = torch.randperm(x.size(0)).to(x.device)
    mixed_x = lam * x + (1 - lam) * x[index]
    return mixed_x, y, y[index], lam
 
def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

In [ ]:
# KIẾN TRÚC SWIN TRANSFORMER
# Dùng Swin-S pretrained ImageNet-22k từ thư viện timm.
#   - swin_small_patch4_window7_224: patch 4x4, window 7x7, input 224x224
#   - pretrained='True': tải trọng số ImageNet-22k
#   - Output: 768 chiều (khác ResNet50 là 2048)
# Head: 768 → 384 → 13
#   - 768→384: giữ nhiều thông tin hơn 768→256, Swin đã học đặc trưng phong phú
#   - Không Softmax vì FocalLoss đã tích hợp LogSoftmax bên trong
 
def build_swin(num_classes):
    # Tải Swin-S pretrained, num_classes=0 để lấy backbone thuần (không có head)
    backbone = timm.create_model(
        "swin_small_patch4_window7_224",
        pretrained=True,
        num_classes=0,      # Bỏ head gốc của timm
        global_pool='avg'   # Global average pooling sang vector 768 chiều
    )
 
    # Lấy số chiều output của backbone = 768 với Swin-S
    in_features = backbone.num_features  # 768
 
    # Bọc backbone + head mới vào một module
    class SwinClassifier(nn.Module):
        def __init__(self, backbone, in_features, num_classes):
            super().__init__()
            self.backbone = backbone
            self.head = nn.Sequential(
                nn.Dropout(p=0.3),
                nn.Linear(in_features, 384),
                nn.ReLU(inplace=True),
                nn.Dropout(p=0.2),
                nn.Linear(384, num_classes)
                # Không Softmax
            )
 
        def forward(self, x):
            features = self.backbone(x)  # (B, 768)
            return self.head(features)   # (B, 13)
 
    return SwinClassifier(backbone, in_features, num_classes)
 
model = build_swin(NUM_CLASSES).to(DEVICE)
 
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nMô hình Swin-S:")
print(f"  Tổng tham số      : {total_params:,}")
print(f"  Tham số trainable : {trainable_params:,}")

In [ ]:
# HÀM FREEZE / UNFREEZE
# Swin-S có 4 stage trong backbone (layers[0]→layers[3]).
#   Warmup   : freeze toàn bộ backbone, chỉ train head
#   Finetune : mở stage2 + stage3 (layers[2] và layers[3])
 
def freeze_backbone(model):
    """Freeze toàn bộ backbone Swin, chỉ train head."""
    for param in model.backbone.parameters():
        param.requires_grad = False
 
def unfreeze_stage2_stage3(model):
    """Mở stage2 (layers[2]) và stage3 (layers[3]) của Swin backbone."""
    # Mở head (luôn train)
    for param in model.head.parameters():
        param.requires_grad = True
    # Mở 2 stage cuối của backbone
    for name, param in model.backbone.named_parameters():
        if "layers.2" in name or "layers.3" in name:
            param.requires_grad = True
 
def count_trainable(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)
 

In [ ]:
# HÀM TRAIN VÀ VALIDATE MỘT EPOCH
# Giữ cấu trúc giống ResNet50 để dễ so sánh và maintain code.
 
criterion = FocalLoss(gamma=FOCAL_GAMMA, label_smoothing=LABEL_SMOOTHING)
 
def train_one_epoch(model, loader, optimizer, use_mixup=True):
    model.train()
    total_loss = total_correct = total_samples = 0
 
    for images, labels in loader:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
 
        if use_mixup:
            images, labels_a, labels_b, lam = mixup_data(images, labels)
            outputs = model(images)
            loss    = mixup_criterion(criterion, outputs, labels_a, labels_b, lam)
        else:
            outputs = model(images)
            loss    = criterion(outputs, labels)
 
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
 
        total_loss    += loss.item() * images.size(0)
        preds          = outputs.argmax(dim=1)
        total_correct += (preds == labels).sum().item()
        total_samples += images.size(0)
 
    return total_loss / total_samples, total_correct / total_samples
 
 
def validate(model, loader):
    model.eval()
    total_loss = total_correct = total_samples = 0
    all_preds  = []
    all_labels = []
 
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)
 
            outputs = model(images)
            loss    = criterion(outputs, labels)
            preds   = outputs.argmax(dim=1)
 
            total_loss    += loss.item() * images.size(0)
            total_correct += (preds == labels).sum().item()
            total_samples += images.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
 
    avg_loss = total_loss / total_samples
    avg_acc  = total_correct / total_samples
    f1_macro = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    return avg_loss, avg_acc, f1_macro, all_preds, all_labels

In [ ]:
# VÒNG LẶP TRAIN CHÍNH
# 2 giai đoạn: Warmup -> Finetune.
 
history = {
    "epoch": [], "stage": [], "train_loss": [], "train_acc": [],
    "val_loss": [], "val_acc": [], "val_f1": [], "lr": [], "epoch_time": [],
}
 
best_val_acc    = 0.0
early_stop_cnt  = 0
best_model_path = os.path.join(OUTPUT_DIR, "swin_best.pth")
 
print("\n" + "=" * 70)
print("BẮT ĐẦU TRAIN SWIN TRANSFORMER")
print("=" * 70)
 
global_epoch = 0
 
for stage_name, n_epochs in [
    ("Warmup",   EPOCHS_WARMUP),
    ("Finetune", EPOCHS_FINETUNE),
]:
    if stage_name == "Warmup":
        freeze_backbone(model)
        # Chỉ train head, backbone hoàn toàn frozen
        optimizer = optim.AdamW(
            filter(lambda p: p.requires_grad, model.parameters()),
            lr=LR_HEAD, weight_decay=WEIGHT_DECAY
        )
        print(f"\n[Giai đoạn 1: Warm-up] "
              f"Freeze backbone | LR head={LR_HEAD} | Epochs={n_epochs}")
 
    else:  # Finetune
        unfreeze_stage2_stage3(model)
        # 2 param groups: backbone LR rất nhỏ để bảo vệ attention weights
        optimizer = optim.AdamW([
            {"params": model.head.parameters(), "lr": LR_HEAD / 10},
            {"params": [p for n, p in model.backbone.named_parameters()
                        if ("layers.2" in n or "layers.3" in n)
                        and p.requires_grad],
             "lr": LR_BACKBONE},
        ], weight_decay=WEIGHT_DECAY)
        print(f"\n[Giai đoạn 2: Fine-tune stage2+stage3] "
              f"LR backbone={LR_BACKBONE} | LR head={LR_HEAD/10} | Epochs={n_epochs}")
 
    print(f"  Tham số trainable: {count_trainable(model):,}")
 
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=n_epochs, eta_min=1e-7
    )
    use_mixup = (stage_name != "Warmup")
 
    for ep in range(n_epochs):
        global_epoch += 1
        t0 = time.time()
 
        train_loss, train_acc = train_one_epoch(
            model, train_loader, optimizer, use_mixup=use_mixup)
        val_loss, val_acc, val_f1, _, _ = validate(model, val_loader)
        scheduler.step()
 
        elapsed = time.time() - t0
        cur_lr  = optimizer.param_groups[0]["lr"]
 
        history["epoch"].append(global_epoch)
        history["stage"].append(stage_name)
        history["train_loss"].append(round(train_loss, 4))
        history["train_acc"].append(round(train_acc, 4))
        history["val_loss"].append(round(val_loss, 4))
        history["val_acc"].append(round(val_acc, 4))
        history["val_f1"].append(round(val_f1, 4))
        history["lr"].append(round(cur_lr, 7))
        history["epoch_time"].append(round(elapsed, 1))
 
        print(f"  Epoch [{global_epoch:02d}/{TOTAL_EPOCHS}] ({stage_name}) | "
              f"Train loss={train_loss:.4f} acc={train_acc:.4f} | "
              f"Val loss={val_loss:.4f} acc={val_acc:.4f} F1={val_f1:.4f} | "
              f"LR={cur_lr:.2e} | {elapsed:.1f}s")
 
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save({
                "epoch"      : global_epoch,
                "stage"      : stage_name,
                "model_state": model.state_dict(),
                "optimizer"  : optimizer.state_dict(),
                "val_acc"    : val_acc,
                "val_f1"     : val_f1,
            }, best_model_path)
            print(f"     Lưu best model (val_acc={val_acc:.4f})")
            early_stop_cnt = 0
        else:
            early_stop_cnt += 1
            if early_stop_cnt >= EARLY_STOP_PATIENCE:
                print(f"\n   Early stopping tại epoch {global_epoch} "
                      f"(không cải thiện {EARLY_STOP_PATIENCE} epochs liên tiếp)")
                break
 
print("\n" + "=" * 70)
print(f"TRAIN HOÀN TẤT | Best val accuracy: {best_val_acc:.4f}")
print("=" * 70)

In [ ]:
#ĐÁNH GIÁ TRÊN TẬP TEST
# Load best model và đánh giá trên tập test độc lập..
 
checkpoint = torch.load(best_model_path, map_location=DEVICE)
model.load_state_dict(checkpoint["model_state"])
print(f"\nLoad best model từ epoch {checkpoint['epoch']} "
      f"(val_acc={checkpoint['val_acc']:.4f})")
 
test_loss, test_acc, test_f1, test_preds, test_labels = validate(
    model, test_loader)
 
print(f"\nKết quả trên tập TEST:")
print(f"  Loss     : {test_loss:.4f}")
print(f"  Accuracy : {test_acc:.4f}  ({test_acc*100:.2f}%)")
print(f"  F1 Macro : {test_f1:.4f}")
 
report = classification_report(
    test_labels, test_preds, target_names=CLASSES, digits=4)
print(f"\nClassification Report:\n{report}")
 

In [ ]:
# XUẤT SỐ LIỆU 
pd.DataFrame(history).to_csv(
    os.path.join(OUTPUT_DIR, "training_history.csv"), index=False)
print("Đã lưu: training_history.csv")
 
report_dict = classification_report(
    test_labels, test_preds, target_names=CLASSES, digits=4, output_dict=True)
pd.DataFrame(report_dict).transpose().round(4).to_csv(
    os.path.join(OUTPUT_DIR, "classification_report.csv"))
print("Đã lưu: classification_report.csv")
 
summary = {
    "model"            : "Swin-S",
    "best_epoch"       : int(checkpoint["epoch"]),
    "best_val_acc"     : round(float(checkpoint["val_acc"]), 4),
    "best_val_f1"      : round(float(checkpoint["val_f1"]), 4),
    "test_loss"        : round(test_loss, 4),
    "test_accuracy"    : round(test_acc, 4),
    "test_accuracy_pct": round(test_acc * 100, 2),
    "test_f1_macro"    : round(test_f1, 4),
    "total_epochs_run" : global_epoch,
    "config": {
        "img_size"        : IMG_SIZE,
        "batch_size"      : BATCH_SIZE,
        "focal_gamma"     : FOCAL_GAMMA,
        "label_smoothing" : LABEL_SMOOTHING,
        "weight_decay"    : WEIGHT_DECAY,
        "epochs_warmup"   : EPOCHS_WARMUP,
        "epochs_finetune" : EPOCHS_FINETUNE,
        "lr_head"         : LR_HEAD,
        "lr_backbone"     : LR_BACKBONE,
    }
}
with open(os.path.join(OUTPUT_DIR, "summary.json"), "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)
print("Đã lưu: summary.json")
 
#Bảng per-class Swin
print("\n" + "=" * 70)
print("BẢNG KẾT QUẢ PER-CLASS SWIN (copy vào luận văn)")
print("=" * 70)
print(f"  {'Lớp':<15} {'Precision':>10} {'Recall':>10} {'F1-score':>10} {'Support':>10}")
print(f"  {'-'*57}")
for cls in CLASSES:
    p  = report_dict[cls]["precision"]
    r  = report_dict[cls]["recall"]
    f1 = report_dict[cls]["f1-score"]
    s  = int(report_dict[cls]["support"])
    print(f"  {cls:<15} {p:>10.4f} {r:>10.4f} {f1:>10.4f} {s:>10}")
print(f"  {'-'*57}")
print(f"  {'Accuracy':<15} {'':>10} {'':>10} {test_acc:>10.4f} {len(test_labels):>10}")
print(f"  {'Macro avg':<15} "
      f"{report_dict['macro avg']['precision']:>10.4f} "
      f"{report_dict['macro avg']['recall']:>10.4f} "
      f"{report_dict['macro avg']['f1-score']:>10.4f} "
      f"{len(test_labels):>10}")
print(f"  {'Weighted avg':<15} "
      f"{report_dict['weighted avg']['precision']:>10.4f} "
      f"{report_dict['weighted avg']['recall']:>10.4f} "
      f"{report_dict['weighted avg']['f1-score']:>10.4f} "
      f"{len(test_labels):>10}")
 
# Bảng so sánh ResNet50 vs Swin per-class
print("\n" + "=" * 70)
print("SO SÁNH F1-SCORE: ResNet50 vs Swin (dùng cho luận văn)")
print("=" * 70)
 
# Kết quả ResNet50 đã có từ notebook trước
resnet_f1 = {
    "BA": 0.9954, "BNE": 0.8656, "EO": 0.9948, "ERB": 0.9933,
    "IG": 1.0000, "LY": 0.9828, "MMY": 0.7960, "MO": 0.9854,
    "MY": 0.7965, "MYELOBLAST": 1.0000, "PLATELET": 0.9954,
    "PMY": 0.7368, "SNE": 0.8435
}
 
print(f"  {'Lớp':<15} {'ResNet50 F1':>12} {'Swin F1':>10} {'Δ':>8} {'Mạnh hơn':>12}")
print(f"  {'-'*60}")
for cls in CLASSES:
    r_f1 = resnet_f1.get(cls, 0.0)
    s_f1 = report_dict[cls]["f1-score"]
    delta = s_f1 - r_f1
    winner = "Swin ↑" if delta > 0.005 else ("ResNet ↑" if delta < -0.005 else "Tương đương")
    print(f"  {cls:<15} {r_f1:>12.4f} {s_f1:>10.4f} {delta:>+8.4f} {winner:>12}")
 
print(f"  {'-'*60}")
resnet_macro = 0.9220
swin_macro   = report_dict['macro avg']['f1-score']
delta_macro  = swin_macro - resnet_macro
print(f"  {'Macro avg':<15} {resnet_macro:>12.4f} {swin_macro:>10.4f} "
      f"{delta_macro:>+8.4f}")

In [ ]:
# VẼ BIỂU ĐỒ
# Xuất 3 biểu đồ:
#   training_curves.png      : Loss/Acc/F1 theo epoch (2 giai đoạn)
#   confusion_matrix.png     : Ma trận nhầm lẫn số lượng + tỉ lệ
#   f1_comparison.png        : Biểu đồ so sánh F1 ResNet50 vs Swin từng lớp
 
stage_colors = {"Warmup": "#FFF3CD", "Finetune": "#D4EDDA"}
 
def shade_stages(ax):
    prev, start = None, 1
    for ep, st in zip(history["epoch"], history["stage"]):
        if st != prev:
            if prev:
                ax.axvspan(start-0.5, ep-0.5, alpha=0.15,
                           color=stage_colors[prev], label=f"Stage: {prev}")
            start, prev = ep, st
    ax.axvspan(start-0.5, history["epoch"][-1]+0.5, alpha=0.15,
               color=stage_colors[prev], label=f"Stage: {prev}")
 
# Training curves
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
epochs_x  = history["epoch"]
 
ax = axes[0]
shade_stages(ax)
ax.plot(epochs_x, history["train_loss"], label="Train Loss", color="#E74C3C", linewidth=2)
ax.plot(epochs_x, history["val_loss"],   label="Val Loss",   color="#3498DB", linewidth=2)
ax.set_title("Loss theo Epoch", fontsize=13, fontweight='bold')
ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
ax.legend(); ax.grid(True, alpha=0.3)
 
ax = axes[1]
shade_stages(ax)
ax.plot(epochs_x, [v*100 for v in history["train_acc"]], label="Train Acc", color="#E74C3C", linewidth=2)
ax.plot(epochs_x, [v*100 for v in history["val_acc"]],   label="Val Acc",   color="#3498DB", linewidth=2)
ax.set_title("Accuracy theo Epoch", fontsize=13, fontweight='bold')
ax.set_xlabel("Epoch"); ax.set_ylabel("Accuracy (%)")
ax.yaxis.set_major_formatter(ticker.FormatStrFormatter('%.1f'))
ax.legend(); ax.grid(True, alpha=0.3)
 
ax = axes[2]
shade_stages(ax)
ax.plot(epochs_x, history["val_f1"], label="Val F1 Macro", color="#27AE60", linewidth=2)
ax.set_title("F1 Macro theo Epoch", fontsize=13, fontweight='bold')
ax.set_xlabel("Epoch"); ax.set_ylabel("F1 Score")
ax.legend(); ax.grid(True, alpha=0.3)
 
plt.suptitle("Swin Transformer — Training Curves", fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "training_curves.png"), dpi=150, bbox_inches='tight')
plt.show()
print("Đã lưu: training_curves.png")
 
#Confusion matrix
cm      = confusion_matrix(test_labels, test_preds)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
 
fig, axes = plt.subplots(1, 2, figsize=(22, 9))
for ax, data, fmt, title in zip(
    axes, [cm, cm_norm], ["d", ".2f"],
    ["Confusion Matrix (Số lượng)", "Confusion Matrix (Tỉ lệ)"]
):
    sns.heatmap(data, annot=True, fmt=fmt,
                xticklabels=CLASSES, yticklabels=CLASSES,
                cmap="Blues", ax=ax, linewidths=0.5, annot_kws={"size": 8})
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel("Predicted", fontsize=11)
    ax.set_ylabel("Actual", fontsize=11)
    ax.tick_params(axis='x', rotation=45)
    ax.tick_params(axis='y', rotation=0)
 
plt.suptitle("Swin Transformer — Confusion Matrix trên tập Test",
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "confusion_matrix.png"), dpi=150, bbox_inches='tight')
plt.show()
print("Đã lưu: confusion_matrix.png")
 
#Biểu đồ so sánh F1 ResNet50 vs Swin
swin_f1_list   = [report_dict[cls]["f1-score"] for cls in CLASSES]
resnet_f1_list = [resnet_f1[cls] for cls in CLASSES]
x = np.arange(len(CLASSES))
w = 0.35
 
fig, ax = plt.subplots(figsize=(14, 6))
bars1 = ax.bar(x - w/2, resnet_f1_list, w, label="ResNet50", color="#3498DB", alpha=0.85)
bars2 = ax.bar(x + w/2, swin_f1_list,   w, label="Swin-S",   color="#E74C3C", alpha=0.85)
 
ax.set_xlabel("Lớp tế bào", fontsize=12)
ax.set_ylabel("F1-score", fontsize=12)
ax.set_title("So sánh F1-score per-class: ResNet50 vs Swin Transformer",
             fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(CLASSES, rotation=30, ha='right')
ax.set_ylim(0, 1.1)
ax.axhline(y=resnet_macro, color="#3498DB", linestyle='--', alpha=0.5,
           label=f"ResNet50 Macro avg ({resnet_macro:.4f})")
ax.axhline(y=swin_macro,   color="#E74C3C", linestyle='--', alpha=0.5,
           label=f"Swin Macro avg ({swin_macro:.4f})")
ax.legend(fontsize=10)
ax.grid(True, axis='y', alpha=0.3)
 
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "f1_comparison.png"), dpi=150, bbox_inches='tight')
plt.show()
print("Đã lưu: f1_comparison.png")

In [ ]:
# TỔNG KẾT OUTPUT
# Kiểm tra tất cả file đã xuất thành công.
 
print("\n" + "=" * 70)
print("TẤT CẢ FILE OUTPUT")
print("=" * 70)
for fname, desc in {
    "swin_best.pth"             : "Best model weights → dùng cho Ensemble",
    "training_history.csv"      : "Loss/Acc/F1 từng epoch → bảng báo cáo",
    "classification_report.csv" : "Precision/Recall/F1 từng lớp → bảng báo cáo",
    "summary.json"              : "Tóm tắt kết quả + cấu hình",
    "training_curves.png"       : "Biểu đồ Loss/Acc/F1 → hình báo cáo",
    "confusion_matrix.png"      : "Ma trận nhầm lẫn → hình báo cáo",
    "f1_comparison.png"         : "So sánh F1 ResNet50 vs Swin → hình báo cáo",
}.items():
    fpath  = os.path.join(OUTPUT_DIR, fname)
    exists = "ok" if os.path.exists(fpath) else "no ok"
    print(f"  {exists} {fname:<35} → {desc}")
 
print(f"\nKết quả cuối cùng Swin-S:")
print(f"  Test Accuracy : {test_acc*100:.2f}%")
print(f"  Test F1 Macro : {test_f1:.4f}")
print(f"  Best epoch    : {checkpoint['epoch']}")
print(f"\n→ Bước tiếp theo: dùng resnet50_best.pth + swin_best.pth cho Ensemble!")